Menyiapkan Dataset
-

In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


Membuat SparkSession
--

In [2]:
# Membuat SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/29 15:05:46 WARN Utils: Your hostname, zaydann resolves to a loopback address: 127.0.1.1; using 192.168.50.47 instead (on interface wlo1)
26/10/29 15:05:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 15:05:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


Membaca transaksi dari HDFS
--

In [3]:
# Membaca data transaksi dari HDFS
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

# Menambahkan kolom pendapatan
df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

# Membuat DataFrame target cabang
df_target = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

# Menampilkan jumlah data
print("Jumlah transaksi:", df_transaksi.count())
print("Jumlah target cabang:", df_target.count())

# Menampilkan contoh data transaksi
df_transaksi.show(5)

# Menampilkan data target cabang
df_target.show()

Jumlah transaksi: 500


Jumlah target cabang: 5
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sar

A. Join & Perbandingan Target
--

In [4]:
# A. Join & Perbandingan Target

# Menghitung total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy(
    "kota"
).agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Menggabungkan ringkasan pendapatan dengan target cabang
hasil_target = ringkasan_kota.join(
    df_target,
    on="kota",
    how="inner"
)

# Menghitung persentase pencapaian target
hasil_target = hasil_target.withColumn(
    "pencapaian_persen",
    (
        col("total_pendapatan") / col("target_bulanan")
    ) * 100
)

# Mengurutkan berdasarkan pencapaian tertinggi
hasil_target = hasil_target.orderBy(
    col("pencapaian_persen").desc()
)

hasil_target.show(truncate=False)

+----------+----------------+--------------+----------+------------------+
|kota      |total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen |
+----------+----------------+--------------+----------+------------------+
|Purworejo |45650000        |30000000      |Fitri     |152.16666666666669|
|Solo      |33475000        |40000000      |Bayu      |83.6875           |
|Yogyakarta|47275000        |60000000      |Joko      |78.79166666666667 |
|Magelang  |31650000        |45000000      |Rani      |70.33333333333334 |
|Semarang  |38175000        |55000000      |Sari      |69.4090909090909  |
+----------+----------------+--------------+----------+------------------+



B. Window Function — Kategori Terlaris Per Kota
-

In [5]:
# B. Window Function — Kategori Terlaris per Kota

# Menghitung total pendapatan setiap kategori pada setiap kota
ringkasan_kategori = df_transaksi.groupBy(
    "kota",
    "kategori"
).agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Membuat window berdasarkan kota
window_kategori = Window.partitionBy(
    "kota"
).orderBy(
    col("total_pendapatan").desc()
)

# Menentukan peringkat setiap kategori pada masing-masing kota
hasil_kategori = ringkasan_kategori.withColumn(
    "peringkat",
    row_number().over(window_kategori)
)

# Mengambil kategori dengan peringkat 1
top_kategori = hasil_kategori.filter(
    col("peringkat") == 1
).orderBy(
    "kota"
)

top_kategori.show(truncate=False)

+----------+----------------------+----------------+---------+
|kota      |kategori              |total_pendapatan|peringkat|
+----------+----------------------+----------------+---------+
|Magelang  |Kesehatan & Kecantikan|7275000         |1        |
|Purworejo |Kesehatan & Kecantikan|10075000        |1        |
|Semarang  |Rumah Tangga          |11125000        |1        |
|Solo      |Kesehatan & Kecantikan|8425000         |1        |
|Yogyakarta|Fashion               |13325000        |1        |
+----------+----------------------+----------------+---------+



C. Spark Sql
-

In [6]:
# C. Spark SQL

# Membuat temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

# Menghitung jumlah transaksi setiap kota menggunakan Spark SQL
hasil_sql = spark.sql("""
    SELECT
        t.kota,
        d.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    INNER JOIN target_cabang d
        ON t.kota = d.kota
    GROUP BY
        t.kota,
        d.pic_cabang
    ORDER BY
        jumlah_transaksi DESC
""")

hasil_sql.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Kesimpulan
-

Berdasarkan hasil analisis, cabang Purworejo memiliki kinerja paling baik jika dilihat dari pencapaian target penjualan. Total pendapatannya mencapai Rp45.650.000 dari target Rp30.000.000, sehingga persentase pencapaiannya sebesar 152,17%. Sementara itu, cabang Semarang perlu mendapatkan perhatian lebih dari manajemen karena pencapaian targetnya paling rendah, yaitu 69,41%, dengan total pendapatan Rp38.175.000 dari target Rp55.000.000. Dari hasil window function, kategori dengan pendapatan tertinggi di cabang Semarang adalah Rumah Tangga dengan total Rp11.125.000. Hasil ini dapat menjadi bahan pertimbangan untuk mengevaluasi strategi penjualan dan meningkatkan pendapatan cabang tersebut. Di sisi lain, kategori Kesehatan & Kecantikan menjadi kategori terlaris di Purworejo dengan pendapatan Rp10.075.000. Informasi tersebut dapat digunakan untuk menentukan strategi promosi dan pengelolaan produk sesuai kondisi setiap cabang.

Menutup SparkSession
--

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
